In [2]:

import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import mlflow
import mlflow.pytorch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# Project paths
PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"

sys.path.append(str(PROJECT_ROOT / "src"))

from data import CLASS_NAMES, DEVICE, BATCH_SIZE
from resnet_data import create_resnet_dataloaders
from resnet_model import ResNet50Classifier

# Configuration
NUM_CLASSES = len(CLASS_NAMES)
EPOCHS = 20
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

MODEL_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("Classes:", CLASS_NAMES)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)


Device: cuda
Classes: ['normal', 'glioma_tumor', 'meningioma_tumor', 'pituitary_tumor']
Batch size: 16
Epochs: 20


In [3]:
from resnet_data import create_resnet_dataloaders

train_loader, val_loader, test_loader = create_resnet_dataloaders()

images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Images shape: torch.Size([16, 3, 256, 256])
Labels shape: torch.Size([16])
Training batches: 125
Validation batches: 27
Testing batches: 27


In [4]:

# Initialize pretrained ResNet50
model = ResNet50Classifier(
    num_classes=NUM_CLASSES,
    pretrained=True
).to(DEVICE)

# Check trainable parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

# Test forward pass
images, labels = next(iter(train_loader))
images = images.to(DEVICE)

with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\pc/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:32<00:00, 3.17MB/s]


Total parameters: 23,516,228
Trainable parameters: 8,196
Input shape: torch.Size([16, 3, 256, 256])
Output shape: torch.Size([16, 4])


In [5]:

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimize trainable parameters only
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001,
    weight_decay=1e-4
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# Training configuration
EPOCHS = 10
PATIENCE = 4

print("Optimizer:", optimizer)
print("Criterion:", criterion)
print("Epochs:", EPOCHS)
print("Early stopping patience:", PATIENCE)


Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)
Criterion: CrossEntropyLoss()
Epochs: 10
Early stopping patience: 4


In [6]:

# Training history
history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss = float("inf")
patience_counter = 0

with mlflow.start_run(run_name="ResNet50_Frozen"):

    # Log parameters
    mlflow.log_params({
        "architecture": "ResNet50",
        "pretrained": True,
        "backbone_frozen": True,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "optimizer": "Adam",
        "loss": "CrossEntropyLoss",
        "scheduler": "ReduceLROnPlateau",
        "early_stopping_patience": PATIENCE
    })

    for epoch in range(EPOCHS):

        # --------------------
        # Training
        # --------------------
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            train_correct += (predictions == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_accuracy = train_correct / train_total

        # --------------------
        # Validation
        # --------------------
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                predictions = outputs.argmax(dim=1)
                val_correct += (predictions == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_accuracy = val_correct / val_total

        # Store metrics
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        # Learning rate scheduler
        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        # Log metrics
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "learning_rate": current_lr
        }, step=epoch + 1)

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f} | "
            f"LR: {current_lr}"
        )

        # Save best model
        if val_loss < best_val_loss:

            best_val_loss = val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                MODEL_DIR / "best_resnet50_frozen.pth"
            )

        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Load best checkpoint
    model.load_state_dict(
        torch.load(
            MODEL_DIR / "best_resnet50_frozen.pth",
            map_location=DEVICE,
            weights_only=True
        )
    )

    # Log best model as MLflow artifact
    mlflow.pytorch.log_model(
        model,
        artifact_path="model"
    )

    print("\nBest Validation Loss:", best_val_loss)
    print("Best model saved and logged to MLflow.")


Epoch [1/10] Train Loss: 0.9923 | Train Acc: 0.6208 | Val Loss: 0.7896 | Val Acc: 0.7307 | LR: 0.001
Epoch [2/10] Train Loss: 0.7331 | Train Acc: 0.7353 | Val Loss: 0.6664 | Val Acc: 0.7728 | LR: 0.001
Epoch [3/10] Train Loss: 0.6446 | Train Acc: 0.7634 | Val Loss: 0.6218 | Val Acc: 0.7681 | LR: 0.001
Epoch [4/10] Train Loss: 0.6197 | Train Acc: 0.7700 | Val Loss: 0.5781 | Val Acc: 0.7775 | LR: 0.001
Epoch [5/10] Train Loss: 0.5747 | Train Acc: 0.7891 | Val Loss: 0.5538 | Val Acc: 0.7916 | LR: 0.001
Epoch [6/10] Train Loss: 0.5530 | Train Acc: 0.7855 | Val Loss: 0.5385 | Val Acc: 0.7752 | LR: 0.001
Epoch [7/10] Train Loss: 0.5351 | Train Acc: 0.8011 | Val Loss: 0.5318 | Val Acc: 0.7892 | LR: 0.001
Epoch [8/10] Train Loss: 0.5343 | Train Acc: 0.7986 | Val Loss: 0.5399 | Val Acc: 0.7845 | LR: 0.001
Epoch [9/10] Train Loss: 0.5052 | Train Acc: 0.8147 | Val Loss: 0.5138 | Val Acc: 0.7963 | LR: 0.001
Epoch [10/10] Train Loss: 0.5040 | Train Acc: 0.8071 | Val Loss: 0.5112 | Val Acc: 0.8009 |

2026/11/11 19:31:24 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/11 19:31:39 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.20.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torchvision==0.20.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/11 19:31:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Best Validation Loss: 0.5112149828490943
Best model saved and logged to MLflow.


In [7]:

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(DEVICE)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(
    all_labels,
    all_predictions,
    target_names=CLASS_NAMES,
    digits=4
))

cm = confusion_matrix(all_labels, all_predictions)

print("Confusion Matrix:")
print(pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
))


                  precision    recall  f1-score   support

          normal     0.8113    0.9149    0.8600        47
    glioma_tumor     0.9167    0.6617    0.7686       133
meningioma_tumor     0.6642    0.7459    0.7027       122
 pituitary_tumor     0.8511    0.9600    0.9023       125

        accuracy                         0.8009       427
       macro avg     0.8108    0.8206    0.8084       427
    weighted avg     0.8137    0.8009    0.7989       427

Confusion Matrix:
                  normal  glioma_tumor  meningioma_tumor  pituitary_tumor
normal                43             0                 3                1
glioma_tumor           5            88                40                0
meningioma_tumor       4             7                91               20
pituitary_tumor        1             1                 3              120


In [8]:

# Freeze the entire backbone first
for param in model.model.parameters():
    param.requires_grad = False

# Unfreeze layer4
for param in model.model.layer4.parameters():
    param.requires_grad = True

# Keep the classifier trainable
for param in model.model.fc.parameters():
    param.requires_grad = True

# Count trainable parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\nTrainable layers:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)


Total parameters: 23,516,228
Trainable parameters: 14,972,932

Trainable layers:
model.layer4.0.conv1.weight
model.layer4.0.bn1.weight
model.layer4.0.bn1.bias
model.layer4.0.conv2.weight
model.layer4.0.bn2.weight
model.layer4.0.bn2.bias
model.layer4.0.conv3.weight
model.layer4.0.bn3.weight
model.layer4.0.bn3.bias
model.layer4.0.downsample.0.weight
model.layer4.0.downsample.1.weight
model.layer4.0.downsample.1.bias
model.layer4.1.conv1.weight
model.layer4.1.bn1.weight
model.layer4.1.bn1.bias
model.layer4.1.conv2.weight
model.layer4.1.bn2.weight
model.layer4.1.bn2.bias
model.layer4.1.conv3.weight
model.layer4.1.bn3.weight
model.layer4.1.bn3.bias
model.layer4.2.conv1.weight
model.layer4.2.bn1.weight
model.layer4.2.bn1.bias
model.layer4.2.conv2.weight
model.layer4.2.bn2.weight
model.layer4.2.bn2.bias
model.layer4.2.conv3.weight
model.layer4.2.bn3.weight
model.layer4.2.bn3.bias
model.fc.1.weight
model.fc.1.bias


In [9]:

# Loss function
criterion = nn.CrossEntropyLoss()

# Separate learning rates
optimizer = optim.Adam(
    [
        {
            "params": model.model.layer4.parameters(),
            "lr": 0.0001
        },
        {
            "params": model.model.fc.parameters(),
            "lr": 0.0005
        }
    ],
    weight_decay=1e-4
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# Fine-tuning configuration
EPOCHS = 15
PATIENCE = 5

print("Backbone LR:", optimizer.param_groups[0]["lr"])
print("Classifier LR:", optimizer.param_groups[1]["lr"])
print("Epochs:", EPOCHS)
print("Early stopping patience:", PATIENCE)


Backbone LR: 0.0001
Classifier LR: 0.0005
Epochs: 15
Early stopping patience: 5


In [10]:

history_finetune = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss = float("inf")
patience_counter = 0

with mlflow.start_run(run_name="ResNet50_FineTuning"):

    mlflow.log_params({
        "architecture": "ResNet50",
        "pretrained": True,
        "unfrozen_layers": "layer4",
        "backbone_lr": 0.0001,
        "classifier_lr": 0.0005,
        "weight_decay": WEIGHT_DECAY,
        "optimizer": "Adam",
        "loss": "CrossEntropyLoss",
        "scheduler": "ReduceLROnPlateau",
        "epochs": EPOCHS,
        "early_stopping_patience": PATIENCE
    })

    for epoch in range(EPOCHS):

        # --------------------
        # Training
        # --------------------
        model.train()

        # Keep frozen BatchNorm layers in evaluation mode
        model.model.bn1.eval()
        model.model.layer1.eval()
        model.model.layer2.eval()
        model.model.layer3.eval()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            train_correct += (predictions == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_accuracy = train_correct / train_total

        # --------------------
        # Validation
        # --------------------
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                predictions = outputs.argmax(dim=1)
                val_correct += (predictions == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_accuracy = val_correct / val_total

        history_finetune["train_loss"].append(train_loss)
        history_finetune["train_accuracy"].append(train_accuracy)
        history_finetune["val_loss"].append(val_loss)
        history_finetune["val_accuracy"].append(val_accuracy)

        scheduler.step(val_loss)

        backbone_lr = optimizer.param_groups[0]["lr"]
        classifier_lr = optimizer.param_groups[1]["lr"]

        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "backbone_lr": backbone_lr,
            "classifier_lr": classifier_lr
        }, step=epoch + 1)

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f} | "
            f"Backbone LR: {backbone_lr:.6f} | "
            f"Classifier LR: {classifier_lr:.6f}"
        )

        # Save best checkpoint
        if val_loss < best_val_loss:

            best_val_loss = val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                MODEL_DIR / "best_resnet50_finetuned.pth"
            )

        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Restore best checkpoint
    model.load_state_dict(
        torch.load(
            MODEL_DIR / "best_resnet50_finetuned.pth",
            map_location=DEVICE,
            weights_only=True
        )
    )

    # Log best model to MLflow
    mlflow.pytorch.log_model(
        model,
        artifact_path="model"
    )

    print("\nBest Validation Loss:", best_val_loss)
    print("Fine-tuned model saved and logged to MLflow.")


Epoch [1/15] Train Loss: 0.4260 | Train Acc: 0.8383 | Val Loss: 0.3781 | Val Acc: 0.8548 | Backbone LR: 0.000100 | Classifier LR: 0.000500
Epoch [2/15] Train Loss: 0.2497 | Train Acc: 0.9116 | Val Loss: 0.3100 | Val Acc: 0.8876 | Backbone LR: 0.000100 | Classifier LR: 0.000500
Epoch [3/15] Train Loss: 0.1822 | Train Acc: 0.9312 | Val Loss: 0.2764 | Val Acc: 0.8923 | Backbone LR: 0.000100 | Classifier LR: 0.000500
Epoch [4/15] Train Loss: 0.1415 | Train Acc: 0.9533 | Val Loss: 0.2863 | Val Acc: 0.9063 | Backbone LR: 0.000100 | Classifier LR: 0.000500
Epoch [5/15] Train Loss: 0.0933 | Train Acc: 0.9674 | Val Loss: 0.3875 | Val Acc: 0.9133 | Backbone LR: 0.000100 | Classifier LR: 0.000500
Epoch [6/15] Train Loss: 0.0839 | Train Acc: 0.9724 | Val Loss: 0.3035 | Val Acc: 0.9157 | Backbone LR: 0.000050 | Classifier LR: 0.000250
Epoch [7/15] Train Loss: 0.0633 | Train Acc: 0.9794 | Val Loss: 0.2899 | Val Acc: 0.9110 | Backbone LR: 0.000050 | Classifier LR: 0.000250
Epoch [8/15] Train Loss: 0.

2026/11/11 19:37:33 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/11 19:37:40 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.20.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torchvision==0.20.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/11 19:37:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Best Validation Loss: 0.2763513205202179
Fine-tuned model saved and logged to MLflow.


In [11]:

model.load_state_dict(
    torch.load(
        MODEL_DIR / "best_resnet50_finetuned.pth",
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(DEVICE)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(
    all_labels,
    all_predictions,
    target_names=CLASS_NAMES,
    digits=4
))

cm = confusion_matrix(all_labels, all_predictions)

print("Confusion Matrix:")
print(pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
))


                  precision    recall  f1-score   support

          normal     0.9216    1.0000    0.9592        47
    glioma_tumor     0.8871    0.8271    0.8560       133
meningioma_tumor     0.8254    0.8525    0.8387       122
 pituitary_tumor     0.9524    0.9600    0.9562       125

        accuracy                         0.8923       427
       macro avg     0.8966    0.9099    0.9025       427
    weighted avg     0.8924    0.8923    0.8918       427

Confusion Matrix:
                  normal  glioma_tumor  meningioma_tumor  pituitary_tumor
normal                47             0                 0                0
glioma_tumor           3           110                19                1
meningioma_tumor       1            12               104                5
pituitary_tumor        0             2                 3              120


In [12]:
# evaluation

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from resnet_model import ResNet50Classifier

# Load the selected model checkpoint
model = ResNet50Classifier(
    num_classes=len(CLASS_NAMES),
    pretrained=False
)

checkpoint_path = MODEL_DIR / "best_resnet50_finetuned.pth"

model.load_state_dict(
    torch.load(checkpoint_path, map_location=DEVICE)
)

model = model.to(DEVICE)
model.eval()

print("Model loaded successfully!")
print(f"Device: {DEVICE}")


C:\Users\pc\AppData\Local\Temp\ipykernel_2908\3995123603.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(checkpoint_path, map_location=DEVICE)


Model loaded successfully!
Device: cuda


In [13]:

y_true = []
y_pred = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(predictions.cpu().numpy())

# Metrics
test_accuracy = accuracy_score(y_true, y_pred)

print(f"Test Accuracy: {test_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4
    )
)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
print(cm)


Test Accuracy: 0.9297

Classification Report:
                  precision    recall  f1-score   support

          normal     0.9184    0.9783    0.9474        46
    glioma_tumor     0.9516    0.8872    0.9183       133
meningioma_tumor     0.8760    0.9187    0.8968       123
 pituitary_tumor     0.9680    0.9680    0.9680       125

        accuracy                         0.9297       427
       macro avg     0.9285    0.9380    0.9326       427
    weighted avg     0.9310    0.9297    0.9298       427


Confusion Matrix:
[[ 45   1   0   0]
 [  1 118  13   1]
 [  2   5 113   3]
 [  1   0   3 121]]


In [14]:

final_model_path = MODEL_DIR / "final_resnet50.pth"

torch.save(
    model.state_dict(),
    final_model_path
)

print(f"Final model saved at: {final_model_path}")

Final model saved at: c:\cs\Projects\cv-projects\Brain_tumor\models\final_resnet50.pth


In [15]:

assert final_model_path.exists()

print(f"File size: {final_model_path.stat().st_size / (1024**2):.2f} MB")
print("Model checkpoint saved successfully!")


File size: 90.02 MB
Model checkpoint saved successfully!
